# 🌪️ NLP with Disaster Tweets
**Goal:** Predict which tweets describe real disasters (binary classification)  
**Metric:** F1 Score  
**Approach:** TF-IDF + Logistic Regression → fine-tuned with BERT embeddings

---

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

print('Libraries loaded ✅')

## 2. Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test  = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
sub   = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')

print(f'Train: {train.shape} | Test: {test.shape}')
train.head()

## 3. Exploratory Data Analysis

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Class balance
train['target'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71','#e74c3c'])
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Not Disaster (0)', 'Disaster (1)'], rotation=0)
axes[0].set_xlabel('')

# Tweet length distribution
train['text_len'] = train['text'].str.len()
test['text_len']  = test['text'].str.len()
axes[1].hist(train[train['target']==0]['text_len'], bins=40, alpha=0.6, color='#2ecc71', label='Not Disaster')
axes[1].hist(train[train['target']==1]['text_len'], bins=40, alpha=0.6, color='#e74c3c', label='Disaster')
axes[1].set_title('Tweet Length by Class')
axes[1].legend()

# Word count
train['word_count'] = train['text'].str.split().str.len()
axes[2].hist(train[train['target']==0]['word_count'], bins=30, alpha=0.6, color='#2ecc71', label='Not Disaster')
axes[2].hist(train[train['target']==1]['word_count'], bins=30, alpha=0.6, color='#e74c3c', label='Disaster')
axes[2].set_title('Word Count by Class')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f'\nMissing values:\n{train.isnull().sum()}')

In [ ]:
# Top keywords for disaster vs non-disaster
top_disaster_kw = (
    train[train['target']==1]['keyword']
    .dropna().value_counts().head(10)
)
top_nondisaster_kw = (
    train[train['target']==0]['keyword']
    .dropna().value_counts().head(10)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
top_disaster_kw.plot(kind='barh', ax=axes[0], color='#e74c3c')
axes[0].set_title('Top Keywords — Disaster Tweets')
top_nondisaster_kw.plot(kind='barh', ax=axes[1], color='#2ecc71')
axes[1].set_title('Top Keywords — Non-Disaster Tweets')
plt.tight_layout()
plt.show()

## 4. Text Preprocessing

In [ ]:
def clean_text(text):
    """Clean tweet text for NLP."""
    text = str(text).lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    # Remove HTML entities
    text = re.sub(r'&amp;|&lt;|&gt;', '', text)
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # Keep hashtag words (remove #)
    text = re.sub(r'#(\w+)', r'\1', text)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
train['clean_text'] = train['text'].apply(clean_text)
test['clean_text']  = test['text'].apply(clean_text)

# Fill missing keyword with empty string
train['keyword'] = train['keyword'].fillna('').str.replace('%20', ' ')
test['keyword']  = test['keyword'].fillna('').str.replace('%20', ' ')

# Combine keyword + clean text for richer input
train['input_text'] = train['keyword'] + ' ' + train['clean_text']
test['input_text']  = test['keyword']  + ' ' + test['clean_text']

print('Example cleaned tweet:')
print('Original :', train['text'].iloc[0])
print('Cleaned  :', train['clean_text'].iloc[0])

## 5. Feature Engineering

In [ ]:
def add_features(df):
    """Add handcrafted NLP features."""
    df = df.copy()
    df['char_count']       = df['text'].str.len()
    df['word_count']       = df['text'].str.split().str.len()
    df['unique_words']     = df['text'].apply(lambda x: len(set(str(x).lower().split())))
    df['has_url']          = df['text'].str.contains(r'http|www', regex=True).astype(int)
    df['has_hashtag']      = df['text'].str.contains('#').astype(int)
    df['hashtag_count']    = df['text'].str.count('#')
    df['has_mention']      = df['text'].str.contains('@').astype(int)
    df['exclamation_count']= df['text'].str.count('!')
    df['question_count']   = df['text'].str.count(r'\?')
    df['capital_ratio']    = df['text'].apply(
        lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1)
    )
    df['has_keyword']      = (df['keyword'] != '').astype(int)
    return df

train = add_features(train)
test  = add_features(test)

META_FEATURES = ['char_count','word_count','unique_words','has_url',
                 'has_hashtag','hashtag_count','has_mention',
                 'exclamation_count','question_count','capital_ratio','has_keyword']

print('Meta features added ✅')
train[META_FEATURES].describe()

## 6. Baseline — TF-IDF + Logistic Regression

In [ ]:
from scipy.sparse import hstack
import scipy.sparse as sp

X = train['input_text'].values
y = train['target'].values
X_test = test['input_text'].values

# TF-IDF on words
tfidf_word = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=2
)

# TF-IDF on characters (captures misspellings & slang)
tfidf_char = TfidfVectorizer(
    max_features=20000,
    analyzer='char_wb',
    ngram_range=(3, 5),
    sublinear_tf=True,
    min_df=3
)

X_word = tfidf_word.fit_transform(X)
X_char = tfidf_char.fit_transform(X)
X_meta = sp.csr_matrix(train[META_FEATURES].values)
X_combined = hstack([X_word, X_char, X_meta])

X_test_word = tfidf_word.transform(X_test)
X_test_char = tfidf_char.transform(X_test)
X_test_meta = sp.csr_matrix(test[META_FEATURES].values)
X_test_combined = hstack([X_test_word, X_test_char, X_test_meta])

print(f'Feature matrix shape: {X_combined.shape}')

In [ ]:
# Cross-validation with Logistic Regression
lr = LogisticRegression(C=5, max_iter=1000, class_weight='balanced', solver='lbfgs')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr, X_combined, y, cv=skf, scoring='f1', n_jobs=-1)

print(f'CV F1 scores: {cv_scores}')
print(f'Mean F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 7. Advanced — Multiple Models + Ensemble

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import SGDClassifier
from sklearn.calibration import CalibratedClassifierCV

# Models to ensemble
models = {
    'LogReg'    : LogisticRegression(C=5, max_iter=1000, class_weight='balanced'),
    'LinearSVC' : CalibratedClassifierCV(LinearSVC(C=0.5, max_iter=2000, class_weight='balanced')),
    'SGD'       : SGDClassifier(loss='modified_huber', alpha=1e-4, max_iter=200,
                                class_weight='balanced', random_state=42),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds  = np.zeros((len(train), len(models)))
test_preds = np.zeros((len(test),  len(models)))

for m_idx, (name, model) in enumerate(models.items()):
    fold_scores = []
    test_fold_preds = np.zeros(len(test))
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_combined, y)):
        X_tr, X_val = X_combined[tr_idx], X_combined[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        
        model.fit(X_tr, y_tr)
        val_pred = model.predict(X_val)
        oof_preds[val_idx, m_idx] = val_pred
        
        score = f1_score(y_val, val_pred)
        fold_scores.append(score)
        test_fold_preds += model.predict(X_test_combined) / skf.n_splits
    
    test_preds[:, m_idx] = test_fold_preds
    print(f'{name:12} | CV F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')

print('\nOOF ensemble complete ✅')

In [ ]:
# Ensemble by majority vote
from scipy.stats import mode

ensemble_oof  = mode(oof_preds, axis=1)[0].ravel().astype(int)
ensemble_test = (test_preds.mean(axis=1) >= 0.5).astype(int)

ensemble_f1 = f1_score(y, ensemble_oof)
print(f'Ensemble OOF F1: {ensemble_f1:.4f}')
print(classification_report(y, ensemble_oof, target_names=['Not Disaster','Disaster']))

## 8. Upgrade — BERT Embeddings (distilbert-base)
> Run this section on Kaggle with GPU enabled for best results

In [ ]:
# Uncomment to use on Kaggle with GPU

# !pip install -q transformers datasets
#
# import torch
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
# from datasets import Dataset
#
# MODEL_NAME = 'distilbert-base-uncased'
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#
# def tokenize(batch):
#     return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)
#
# train_ds = Dataset.from_dict({'text': train['input_text'].tolist(), 'label': y.tolist()})
# test_ds  = Dataset.from_dict({'text': test['input_text'].tolist()})
#
# train_ds = train_ds.map(tokenize, batched=True)
# test_ds  = test_ds.map(tokenize, batched=True)
#
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
#
# training_args = TrainingArguments(
#     output_dir='./results',
#     num_train_epochs=3,
#     per_device_train_batch_size=32,
#     per_device_eval_batch_size=64,
#     warmup_steps=200,
#     weight_decay=0.01,
#     evaluation_strategy='epoch',
#     load_best_model_at_end=True,
#     metric_for_best_model='f1',
# )
#
# def compute_metrics(pred):
#     preds = pred.predictions.argmax(-1)
#     return {'f1': f1_score(pred.label_ids, preds)}
#
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_ds,
#     eval_dataset=train_ds,
#     compute_metrics=compute_metrics,
# )
# trainer.train()

print('BERT section ready — uncomment on Kaggle with GPU to run 🚀')

## 9. Generate Submission

In [ ]:
# Use ensemble predictions
submission = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')
submission['target'] = ensemble_test
submission.to_csv('submission.csv', index=False)

print('Submission saved ✅')
print(f'Predicted disaster tweets: {ensemble_test.sum()} / {len(ensemble_test)}')
submission.head(10)

## 10. Error Analysis
> Understanding what the model gets wrong helps you improve

In [ ]:
# Fit final model on all training data
best_model = LogisticRegression(C=5, max_iter=1000, class_weight='balanced')
best_model.fit(X_combined, y)
train['pred'] = best_model.predict(X_combined)

# False Positives — model says disaster, actually not
fp = train[(train['pred']==1) & (train['target']==0)]
print('=== FALSE POSITIVES (predicted disaster, actually not) ===')
print(fp['text'].head(5).to_string())

print('\n=== FALSE NEGATIVES (missed real disasters) ===')
fn = train[(train['pred']==0) & (train['target']==1)]
print(fn['text'].head(5).to_string())

---
## Summary

| Step | Technique | Expected F1 |
|------|-----------|-------------|
| Baseline | TF-IDF (word) + LogReg | ~0.78 |
| Better | TF-IDF (word+char) + meta features | ~0.80 |
| Advanced | Ensemble (LogReg + SVC + SGD) | ~0.82 |
| Best | Fine-tuned DistilBERT | ~0.84–0.86 |

**Key takeaways:**
- Character-level TF-IDF captures slang and misspellings
- Keywords are strong signal — always include them
- BERT gives a meaningful boost for NLP tasks
- Error analysis reveals metaphorical/sarcastic tweets as the hardest cases